# ユニバースの銘柄数の検証:TOPIX 構成銘柄のうち、どこで何銘柄が落ちているか

`rmw18_factor_analysis.ipynb` の「4-3. ユニバースの銘柄数」で、TOPIX ユニバースにしては銘柄数が少ないため、
**6月末時点の TOPIX 構成銘柄(`get_topix_data`)から出発して、分析の各段階で何銘柄が落ちたか** を、理由ごとに集計します。

0〜3章は `rmw18_factor_analysis.ipynb` と同じコードです(設定・取得関数・データ整形・期待収益性の推定)。
同じ処理をたどることで、最後の段階の銘柄数が 4-3 の「全体」と一致するようにしています。
検証の本体は「9. 銘柄が落ちる理由の検証」以降です。

## 0. 設定

In [ ]:
import calendar
import math
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 図の日本語表示

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

try:
    display
except NameError:  # Jupyter 以外で実行したとき
    display = print

# ===== 分析期間 =====
START_MONTH = "1995-07"              # 分析開始月(この月以降のファクターリターンで評価)
END_MONTH = None                     # 分析終了月("2024-12" など。None ならデータの最後まで)

# ===== ポートフォリオ =====
FORMATION_MONTH = 6                  # 毎年この月の月末に組成し、翌月から12か月保有
BIG_CUM_SHARE = 0.90                 # 大型:時価総額の大きい順に足して累積 90% に達するまで
BM_BPS = (0.3, 0.7)                  # B/M の分位点
PROF_BPS = (0.3, 0.7)                # 収益性の分位点
WEIGHTING = "vw"                     # "vw":時価総額加重バイアンドホールド、"ew":等ウェイトバイアンドホールド
CUM_METHOD = "index100"              # 図の累積:"index100" = 100 × Π(1 + r)、"simple" = Π(1 + r) − 1、"log" = Σ log(1 + r)
UNIVERSE_TOPIX_ONLY = True           # True:組成日時点の TOPIX 構成銘柄だけを対象にする
EXCLUDE_IND33 = {"4050", "7050", "7100", "7150", "7200"}  # 電気・ガス、銀行、証券、保険、その他金融

# ===== 収益性指標 =====
MEASURES = ["CBOP", "CP", "ROE", "ROA"]
SCALE = {"CBOP": "asset", "ROA": "asset", "CP": "equity", "ROE": "equity"}

# ===== データ =====
MISSING_SENTINEL = -999999998        # IDSQE 系の欠損値(これ以下の値はすべて欠損扱い)
CTYPE_PRIORITY = ["1", "3", "2"]     # 会計基準の優先順位(1: SEC, 3: IFRS, 2: 日本)
CTYPE_NAME = {"1": "SEC", "2": "日本", "3": "IFRS"}
ACCOUNTING_UNIT = 1e6                # 財務データの単位(百万円)→ 円に直す倍率
AVAIL_FALLBACK_MONTHS = 2            # 発表日・短信入手日が欠損のとき、期末 + 2か月で入手可能とみなす
MAX_STALENESS_MONTHS = 15            # 組成時点から見て、期末がこれより古い決算は使わない

# ===== 推定・検定 =====
WINSOR = (0.01, 0.99)
MICRO_PCT = 0.20                     # 期待収益性の推定から除くマイクロキャップ(ユニバースの時価総額 20 パーセンタイル未満)
MIN_OBS_CS = 30
NW_LAGS = 12

FF_PATH = Path("data/Japan_5_Factors.csv")
CACHE_PATH = Path("cache_raw_data.pkl")
OUTPUT_DIR = Path("outputs_rmw18")
OUTPUT_DIR.mkdir(exist_ok=True)

## 0-1. DB 接続

`temp1.ipynb` で使っている DB 接続用の関数 `fetch_data` と `get_data` を、このセルで import するか定義してください。
キャッシュ(`cache_raw_data.pkl`)がある場合は DB に接続しないので、未定義でも実行できます。

In [ ]:
# 例:from db_utils import fetch_data, get_data
# ↑ temp1.ipynb と同じ import 文・定義をここに書く

if "fetch_data" not in globals() or "get_data" not in globals():
    print("[注意] fetch_data / get_data が未定義です。キャッシュがない場合、データ取得でエラーになります。")

## 0-2. データ取得関数

`temp1.ipynb` の取得関数をそのまま載せています(`get_topix_data` の SELECT のカンマ抜けだけ修正)。
欠損値(-999999998)の置き換えは1章で行います。

In [ ]:
# PL項目の取得
def get_pl_data():
    PL_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[GPLS]         -- 売上総利益
        ,[OPINCM]       -- 営業利益
        ,[CHAGE]        -- 支払利息
        ,[PLCURREN]     -- 為替換算調整勘定，のちのちの分析で必要かもなので一応
        ,[NETINCM]      -- 当期純利益
        ,[RD]           -- 研究開発費
    FROM [IDSQE].[dbo].[NRIFIN_CON_PL4]
    """
    df = fetch_data(PL_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# BS項目の取得
def get_bs_data():
    BS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[CASH]         -- 現金・預金，のちのちの分析で必要かもなので一応
        ,[ACRC]         -- 売上債権，アクルーアルの計算要
        ,[INVENT]       -- 棚卸資産
        ,[INVSCRT]      -- 投資有価証券，別の分析でのちほど用いる
        ,[TOTALBS]      -- 資産合計
        ,[ACPY]         -- 買入債務
        ,[CAPSTTL]      -- 株主資本合計
        ,[NTASTTL]      -- 純資産合計，株主資本合計が欠損しているときに用いる
        ,[EQTY]         -- 資本合計，株主資本合計が欠損しているとき
    FROM [IDSQE].[dbo].[NRIFIN_CON_BS4]
    """
    df = fetch_data(BS_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 決算詳細
def get_accounting_details():
    DETAILS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]            -- 決算期 int yyyymm
        ,[CTYPE]            -- 会計基準 1: SEC式 2: 日本式 3: IFRS式
        ,[SETTLEMENT_FLG]   -- 本中区分 1: 本決算 2: 中間決算 3: 第1四半期決算 4: 第3四半期決算
        ,[FDATE]            -- 決算年月日 char yyyymmdd
        ,[NMONTH]           -- 決算月数 通常12
        ,[ANNOUNCE]         -- 決算発表日，[RCVD]より欠損(-999999998)が多い
        ,[RCVD]             -- 決算短信入手日
    FROM [IDSQE].[dbo].[NRIFIN_CON_AT4]
    """
    df = fetch_data(DETAILS_QUERY, 'idsqe')
    # 欠損値は-999999998
    # 決算発表、短信入手日どちらも欠損している場合は、決算期プラス2か月する（日本の法律）
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 会社属性
# NRI_CODE -> 証券コード
def get_nri_to_ticker_table():
    TABLE_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,[ITEM_DATA]        -- 証券コード
    FROM [IDSQE].[dbo].[COMPANY_CODE]
    WHERE ITEM = 'OLDCD'            -- 証券コード
    """
    df = fetch_data(TABLE_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 業種コード
def get_industry():
    IND_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,c.[ITEM_DATA]
        ,n.[IND_NAME]
    FROM [IDSQE].[dbo].[COMPANY_INDUSTRY_CLASS] c
    JOIN [IDSQE].[dbo].[INDUSTRY_CLASS_NAME] n ON c.ITEM = n.IND
    WHERE [ITEM] = 'IND33'   -- 東証33業種
    """
    df = fetch_data(IND_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# TOPIX構成銘柄
def get_topix_data():
    query = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]
        ,[TO_DATE]
        ,[ITEM]
        ,[ITEM_DATA]
    FROM [IDSQE].[dbo].[STOCK_ISSUE_INDEX]
    WHERE [ITEM] = 'TOPIX';
    """
    return get_data(query, "idsqe")


# 時価総額の取得
def get_market_cap_data():
    # 月次時価総額と、各月末時点で有効な証券コードを取得
    query = """
    SELECT
        prc.[DATE] AS [date],
        prc.[CAP] AS market_cap,
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_PRC] AS prc
    CROSS APPLY (
        -- YYYYMMから月末日（YYYYMMDD）を作成
        SELECT CONVERT(
            int,
            CONVERT(
                char(8),
                EOMONTH(DATEFROMPARTS(
                    prc.[DATE] / 100,
                    prc.[DATE] % 100,
                    1
                )),
                112
            )
        ) AS month_end
    ) AS d
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON prc.[BID] = id.[BID]
        AND id.[IDTYPE] = 'LOCALID'
        AND id.[FDATE] <= d.[month_end]
        AND (id.[TDATE] >= d.[month_end] OR id.[TDATE] IS NULL);
    """
    df = get_data(query, "riskmodels")

    # 証券コードと数値列を整形
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    df["date"] = pd.to_numeric(df["date"], errors="raise").astype(int)
    df["market_cap"] = (
        pd.to_numeric(df["market_cap"], errors="raise")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )
    # 月次Period列を追加
    df["month"] = pd.to_datetime(df["date"].astype(str), format="%Y%m").dt.to_period("M")
    return df


# 銘柄リターンの取得
def get_return_data():
    # 月次リターンと、各月初時点で有効な証券コードを取得
    query = """
    SELECT
        rtn.[DATE],
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code,
        rtn.[RTN]
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_RTN] AS rtn
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON rtn.[BID] = id.[BID]
    WHERE id.[IDTYPE] = 'LOCALID'
      AND TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
          >= TRY_CONVERT(date, CONVERT(varchar(8), id.[FDATE]), 112)
      AND (
          id.[TDATE] IS NULL
          OR TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
             <= TRY_CONVERT(date, CONVERT(varchar(8), id.[TDATE]), 112)
      );
    """
    df = get_data(query, "riskmodels")

    # 証券コード・年月を整形し、リターンを%から小数へ変換
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    dates = pd.to_numeric(df["DATE"], errors="raise").astype(int)
    df["month"] = pd.to_datetime(dates.astype(str), format="%Y%m").dt.to_period("M")
    df["ret"] = pd.to_numeric(df["RTN"], errors="raise").astype(float) / 100
    return df[["security_code", "month", "ret"]].copy()

## 0-3. データ読み込み

上の取得関数で DB からデータを取得し、キャッシュ(`cache_raw_data.pkl`)に保存します。
2回目以降はキャッシュを読みます。取り直したいときは `USE_CACHE = False` にしてください。

In [ ]:
USE_CACHE = True


def load_raw_data(use_cache=USE_CACHE):
    if "取得データ" in globals():  # temp1.ipynb の取得結果がカーネルに残っている場合
        print("既存の `取得データ` を使います")
        return globals()["取得データ"]
    if use_cache and CACHE_PATH.exists():
        print(f"キャッシュ {CACHE_PATH} を読み込みます")
        with open(CACHE_PATH, "rb") as f:
            return pickle.load(f)
    funcs = {
        "PL": get_pl_data, "BS": get_bs_data, "決算詳細": get_accounting_details,
        "証券コード対応表": get_nri_to_ticker_table, "業種": get_industry,
        "TOPIX構成銘柄": get_topix_data, "時価総額": get_market_cap_data, "月次リターン": get_return_data,
    }
    data = {}
    for name, fn in funcs.items():
        print(f"取得中:{name}")
        data[name] = fn()
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(data, f)
    return data


raw = load_raw_data()
for k, v in raw.items():
    print(f"{k}: {v.shape}")

### 共通の関数

In [ ]:
def clean_idsqe(df):
    """IDSQE 系テーブルの整形:欠損値(-999999998 以下)を NaN に、文字列の前後空白を除去"""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].mask(df[c] <= MISSING_SENTINEL)
        elif pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df


def norm_code(s):
    """'2'・2・2.0 のどれで来ても '2' という文字列にそろえる"""
    num = pd.to_numeric(s, errors="coerce")
    out = s.astype(str).str.strip()
    return out.where(num.isna(), num.round().astype("Int64").astype(str))


def norm_sec(s):
    return s.astype(str).str.strip().str.upper()


def to_date(s):
    """yyyymmdd の整数 → Timestamp(0・99999999・欠損は NaT)"""
    v = pd.to_numeric(s, errors="coerce")
    v = v.where((v > 19000000) & (v < 21000000))
    return pd.to_datetime(v.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")


def to_midx(s):
    """月(Period・'YYYY-MM'・yyyymm 整数)→ 通し月番号 year*12 + month - 1"""
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype(int)
        return (v // 100) * 12 + v % 100 - 1
    dt = pd.to_datetime(s.astype(str))
    return dt.dt.year * 12 + dt.dt.month - 1


def month_str_to_midx(m):
    ts = pd.Timestamp(m)
    return ts.year * 12 + ts.month - 1


def midx_to_ts(m):
    m = np.asarray(m, dtype=int)
    return pd.to_datetime({"year": m // 12, "month": m % 12 + 1, "day": 1}) + pd.offsets.MonthEnd(0)


def formation_date(y):
    return pd.Timestamp(int(y), FORMATION_MONTH, calendar.monthrange(int(y), FORMATION_MONTH)[1])


def formation_int(y):
    d = formation_date(y)
    return d.year * 10000 + d.month * 100 + d.day


def winsorize_by(df, cols, by, limits=WINSOR):
    df = df.copy()
    for c in cols:
        g = df.groupby(by)[c]
        lo = g.transform(lambda x: x.quantile(limits[0]))
        hi = g.transform(lambda x: x.quantile(limits[1]))
        df[c] = df[c].clip(lo, hi)
    return df


def ols_nw(y, X, lags=NW_LAGS):
    """OLS(定数項あり)と Newey-West の t 値。X が (T, 0) なら平均の検定"""
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float).reshape(len(y), -1)
    X = np.column_stack([np.ones(len(y)), X])
    ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    y, X = y[ok], X[ok]
    T = len(y)
    if T <= X.shape[1] + 1:
        return np.full(X.shape[1], np.nan), np.full(X.shape[1], np.nan), T
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    u = y - X @ b
    Xu = X * u[:, None]
    S = Xu.T @ Xu
    for l in range(1, min(lags, T - 1) + 1):
        w = 1 - l / (lags + 1)
        G = Xu[l:].T @ Xu[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    return b, b / np.sqrt(np.diag(V)), T


def mean_t(x, lags=NW_LAGS):
    x = pd.Series(x).dropna()
    b, t, T = ols_nw(x.values, np.empty((len(x), 0)), lags)
    return b[0], t[0], T


def p_one_sided(t):
    """平均 > 0 の片側 p 値(正規近似)"""
    return 0.5 * math.erfc(t / math.sqrt(2)) if pd.notna(t) else np.nan


def p_two_sided(t):
    return math.erfc(abs(t) / math.sqrt(2)) if pd.notna(t) else np.nan


def ols_cs(y, X):
    Xc = np.column_stack([np.ones(len(y)), X])
    b, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    return b


def asof_lookup(keys, table, value_cols, date_col):
    """NRI_CODE と日付(yyyymmdd 整数)で、FROM_DATE ≤ 日付 ≤ TO_DATE の行を引く"""
    k = keys[["NRI_CODE", date_col]].drop_duplicates()
    m = k.merge(table, on="NRI_CODE", how="inner")
    m = m[(m["FROM_DATE"] <= m[date_col]) & (m[date_col] <= m["TO_DATE"])]
    m = m.sort_values("FROM_DATE").drop_duplicates(["NRI_CODE", date_col], keep="last")
    return m[["NRI_CODE", date_col] + value_cols]


START_MIDX = month_str_to_midx(START_MONTH)
END_MIDX = month_str_to_midx(END_MONTH) if END_MONTH else None
print(f"分析期間:{START_MONTH} 〜 {END_MONTH or 'データの最後'}")

## 1. 財務データの整形と収益性指標の計算

1. 欠損値を NaN に置き換え、**本決算・決算月数12** の行だけを残す
2. PL・BS・決算詳細を (NRI_CODE, 決算期, 会計基準) で結合(会計基準ごとに別の行のまま)
3. 前期・翌期の値は (NRI_CODE, 会計基準, 決算期 ± 12か月) で結合 → 分子と分母の会計基準が一致
4. 入手可能日から、その決算を使う組成年(6月末までに公表されていればその年)を決める

In [ ]:
KEY = ["NRI_CODE", "FTERM", "CTYPE"]

pl = clean_idsqe(raw["PL"])
bs = clean_idsqe(raw["BS"])
at = clean_idsqe(raw["決算詳細"])
for d in (pl, bs, at):
    d["NRI_CODE"] = norm_sec(d["NRI_CODE"])
    d["CTYPE"] = norm_code(d["CTYPE"])
    d["FTERM"] = pd.to_numeric(d["FTERM"]).astype(int)
at["SETTLEMENT_FLG"] = norm_code(at["SETTLEMENT_FLG"])
at = at[(at["SETTLEMENT_FLG"] == "1") & (at["NMONTH"] == 12)]


def dedupe(df, name):
    n = df.duplicated(KEY).sum()
    if n:
        print(f"[注意] {name}: (NRI_CODE, 決算期, 会計基準) の重複が {n} 行 → 先頭の行を残します")
        df = df.drop_duplicates(KEY)
    return df


at, pl, bs = dedupe(at, "決算詳細"), dedupe(pl, "PL"), dedupe(bs, "BS")
acct = at.merge(pl, on=KEY, how="inner").merge(bs, on=KEY, how="inner")
print(f"本決算・12か月の財務データ:{len(acct):,} 行")

# 決算期、期末日、入手可能日
acct["fy_idx"] = (acct["FTERM"] // 100) * 12 + acct["FTERM"] % 100 - 1
fterm_end = midx_to_ts(acct["fy_idx"].to_numpy())
acct["fdate"] = to_date(acct["FDATE"]).fillna(pd.Series(fterm_end.values, index=acct.index))
ann = to_date(acct["ANNOUNCE"])
rcv = to_date(acct["RCVD"])
fallback = acct["fdate"] + pd.offsets.MonthEnd(AVAIL_FALLBACK_MONTHS)
acct["avail_date"] = ann.where(ann >= acct["fdate"]).fillna(rcv.where(rcv >= acct["fdate"])).fillna(fallback)

# 自己資本・総資産
acct["TA"] = acct["TOTALBS"].where(acct["TOTALBS"] > 0)
be_raw = acct["CAPSTTL"].fillna(acct["EQTY"]).fillna(acct["NTASTTL"])
acct["BE"] = be_raw.where(be_raw > 0)

# 前期(同じ会計基準)の値
PREV_COLS = ["TA", "BE", "CAPSTTL", "EQTY", "NTASTTL", "ACRC", "INVENT", "ACPY"]
prev = acct[["NRI_CODE", "CTYPE", "fy_idx"] + PREV_COLS].copy()
prev["fy_idx"] += 12
prev["prev_exists"] = True
acct = acct.merge(prev.rename(columns={c: c + "_p" for c in PREV_COLS}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
acct["prev_exists"] = acct["prev_exists"].eq(True)


def delta_item(cur, prv, exists):
    d = cur - prv
    d = d.mask(cur.isna() & prv.isna(), 0.0)
    return d.where(exists)


acct["accrual"] = (delta_item(acct["ACRC"], acct["ACRC_p"], acct["prev_exists"])
                   + delta_item(acct["INVENT"], acct["INVENT_p"], acct["prev_exists"])
                   - delta_item(acct["ACPY"], acct["ACPY_p"], acct["prev_exists"]))
int_exp = acct["CHAGE"].fillna(0.0)
rd = acct["RD"].fillna(0.0)
numer = {
    "CBOP": acct["OPINCM"] + rd - acct["accrual"],
    "CP": acct["OPINCM"] - int_exp - acct["accrual"],
    "ROE": acct["NETINCM"],
    "ROA": acct["OPINCM"],
}
for m in MEASURES:
    denom = acct["TA_p"] if SCALE[m] == "asset" else acct["BE_p"]
    acct[m] = numer[m] / denom.where(denom > 0)

# 成長率(自己資本は両期で同じ項目がそろうものだけ)
acct["g_TA"] = acct["TA"] / acct["TA_p"] - 1
g_be = pd.Series(np.nan, index=acct.index)
for col in ["CAPSTTL", "EQTY", "NTASTTL"]:
    ok = (acct[col] > 0) & (acct[col + "_p"] > 0) & g_be.isna()
    g_be[ok] = acct.loc[ok, col] / acct.loc[ok, col + "_p"] - 1
acct["g_BE"] = g_be

# 1期前と1期後(同じ会計基準)
LAG_COLS = MEASURES + ["g_TA", "g_BE"]
lag = acct[["NRI_CODE", "CTYPE", "fy_idx"] + LAG_COLS].copy()
lag["fy_idx"] += 12
acct = acct.merge(lag.rename(columns={c: c + "_lag" for c in LAG_COLS}), on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
lead = acct[["NRI_CODE", "CTYPE", "fy_idx", "avail_date"] + MEASURES].copy()
lead["fy_idx"] -= 12
acct = acct.merge(lead.rename(columns={c: c + "_lead" for c in MEASURES + ["avail_date"]}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
for m in MEASURES:
    acct["d" + m] = acct[m] - acct[m + "_lag"]

# 組成年:入手可能日が組成月末までなら、その年の組成で使える
acct = acct[acct["avail_date"].notna()].copy()
av = acct["avail_date"]
acct["form_year"] = (av.dt.year + (av.dt.month > FORMATION_MONTH).astype(int)).astype(int)
acct["stale_ok"] = (acct["form_year"] * 12 + FORMATION_MONTH - 1 - acct["fy_idx"]) <= MAX_STALENESS_MONTHS
acct["prio"] = acct["CTYPE"].map({c: i for i, c in enumerate(CTYPE_PRIORITY)}).fillna(99)

print("指標ごとの計算できた件数(会計基準別)")
display(acct.groupby(acct["CTYPE"].map(CTYPE_NAME).fillna(acct["CTYPE"]))[MEASURES].count().T)

## 2. 組成年への割り当て・会計基準の選択・市場データとの結合

- 各企業・各組成年について、使える決算のうち最新の決算期を選び、その中で必要な変数(収益性・その変化・投資・自己資本)が
  **同じ会計基準ですべて計算できる** 行を、会計基準の優先順位で1つ選ぶ(指標ごと)
- 組成日(6月末)時点の証券コード・業種・TOPIX 構成を引き、6月末の時価総額と決算期末の時価総額を付ける
- ユニバース:TOPIX 構成銘柄(`UNIVERSE_TOPIX_ONLY`)のうち、電気・ガス・金融を除く銘柄

In [ ]:
code_tbl = clean_idsqe(raw["証券コード対応表"])
code_tbl["NRI_CODE"] = norm_sec(code_tbl["NRI_CODE"])
code_tbl["sec"] = norm_sec(code_tbl["ITEM_DATA"])
code_tbl["FROM_DATE"] = code_tbl["FROM_DATE"].fillna(0)
code_tbl["TO_DATE"] = code_tbl["TO_DATE"].fillna(99999999)

ind_tbl = clean_idsqe(raw["業種"])
ind_tbl["NRI_CODE"] = norm_sec(ind_tbl["NRI_CODE"])
ind_tbl["ind33"] = norm_code(ind_tbl["ITEM_DATA"]).str.zfill(4)
ind_tbl = ind_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE", "ind33"]].drop_duplicates()
ind_tbl["FROM_DATE"] = ind_tbl["FROM_DATE"].fillna(0)
ind_tbl["TO_DATE"] = ind_tbl["TO_DATE"].fillna(99999999)

topix_tbl = clean_idsqe(raw["TOPIX構成銘柄"])
topix_tbl["NRI_CODE"] = norm_sec(topix_tbl["NRI_CODE"])
topix_tbl = topix_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE"]].drop_duplicates()
topix_tbl["FROM_DATE"] = topix_tbl["FROM_DATE"].fillna(0)
topix_tbl["TO_DATE"] = topix_tbl["TO_DATE"].fillna(99999999)
topix_tbl["topix"] = True

mcap = raw["時価総額"].copy()
mcap["sec"] = norm_sec(mcap["security_code"])
mcap["midx"] = to_midx(mcap["month"]) if "month" in mcap else to_midx(mcap["date"])
mcap["me"] = pd.to_numeric(mcap["market_cap"], errors="coerce")
mcap = mcap.loc[mcap["me"] > 0, ["sec", "midx", "me"]].drop_duplicates(["sec", "midx"])

ret = raw["月次リターン"].copy()
ret["sec"] = norm_sec(ret["security_code"])
ret["midx"] = to_midx(ret["month"])
ret["ret"] = pd.to_numeric(ret["ret"], errors="coerce")
ret = ret.loc[ret["ret"].notna(), ["sec", "midx", "ret"]].drop_duplicates(["sec", "midx"])


def select_records(req_cols):
    d = acct[acct["stale_ok"]].dropna(subset=req_cols)
    d = d.sort_values(["NRI_CODE", "form_year", "fy_idx", "prio"], ascending=[True, True, False, True])
    return d.drop_duplicates(["NRI_CODE", "form_year"], keep="first").copy()


def attach_market(d):
    d = d.copy()
    d["F_int"] = d["form_year"].map(formation_int)
    d = d.merge(asof_lookup(d, code_tbl, ["sec"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, ind_tbl, ["ind33"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, topix_tbl, ["topix"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d["topix"] = d["topix"].eq(True)
    d["midx_form"] = d["form_year"] * 12 + FORMATION_MONTH - 1
    d = d.merge(mcap.rename(columns={"midx": "midx_form", "me": "ME_form"}), on=["sec", "midx_form"], how="left")
    d = d.merge(mcap.rename(columns={"midx": "fy_idx", "me": "ME_fye"}), on=["sec", "fy_idx"], how="left")
    d = d[d["sec"].notna() & (d["ME_form"] > 0) & d["ind33"].notna() & ~d["ind33"].isin(EXCLUDE_IND33)].copy()
    if UNIVERSE_TOPIX_ONLY:
        d = d[d["topix"]].copy()
    d["lnME"] = np.log(d["ME_form"])
    bm = d["BE"] * ACCOUNTING_UNIT / d["ME_fye"]
    d["lnBM"] = np.log(bm.where(bm > 0))
    d["micro"] = d["ME_form"] < d.groupby("form_year")["ME_form"].transform(lambda s: s.quantile(MICRO_PCT))
    return d.drop_duplicates(["NRI_CODE", "form_year"])


def build_measure_panel(m):
    inv_col = "g_TA_lag" if SCALE[m] == "asset" else "g_BE_lag"
    d = select_records([m, "d" + m, inv_col, "BE"])
    d = d.rename(columns={m: "Y", "d" + m: "dY", m + "_lead": "Y_lead",
                          "avail_date_lead": "avail_lead", inv_col: "inv"})
    keep = ["NRI_CODE", "form_year", "CTYPE", "fy_idx", "BE", "Y", "dY", "Y_lead", "avail_lead", "inv"]
    d = attach_market(d[keep]).dropna(subset=["lnBM"])
    return winsorize_by(d, ["Y", "dY", "inv", "lnBM"], "form_year")


panels = {}
for m in MEASURES:
    panels[m] = build_measure_panel(m)
    p = panels[m]
    print(f"{m}: {len(p):,} 企業年、組成年 {p['form_year'].min()}〜{p['form_year'].max()}、"
          f"会計基準 {p['CTYPE'].map(CTYPE_NAME).value_counts().to_dict()}")

## 3. 期待収益性の推定

組成年 y の期待収益性 Yhat は、前年のコホート(y−1 年組成)のうち **y 年6月末までに翌期の実績が公表された企業** で
翌期の収益性を回帰し、その係数を y 年の説明変数に当てはめて作ります(先読みなし)。

下の表は、年ごとの回帰の係数の時系列平均と Newey-West t 値(ラグ3)、決定係数 R²・自由度調整済み R² の平均、平均銘柄数です。

In [ ]:
XCOLS = ["Y", "dY", "lnBM", "lnME", "inv"]
NW_LAGS_ANNUAL = 3


def add_expected(p):
    p = p.copy()
    coef_rows = {}
    for y in sorted(p["form_year"].unique()):
        tr = p[(p["form_year"] == y - 1) & ~p["micro"] & (p["avail_lead"] <= formation_date(y))]
        tr = tr.dropna(subset=XCOLS + ["Y_lead"])
        if len(tr) < MIN_OBS_CS:
            continue
        lo, hi = tr["Y_lead"].quantile(WINSOR)
        yv = tr["Y_lead"].clip(lo, hi).to_numpy(float)
        X = tr[XCOLS].to_numpy(float)
        b = ols_cs(yv, X)
        e = yv - (b[0] + X @ b[1:])
        n, k = len(yv), len(XCOLS) + 1
        r2 = 1 - (e @ e) / ((yv - yv.mean()) @ (yv - yv.mean()))
        coef_rows[y] = np.r_[b, r2, 1 - (1 - r2) * (n - 1) / (n - k), n]
    if not coef_rows:
        raise ValueError("期待収益性を推定できる年がありません")
    coef = pd.DataFrame.from_dict(coef_rows, orient="index",
                                  columns=["const"] + XCOLS + ["R2", "adjR2", "N"]).sort_index()
    p["Yhat"] = np.nan
    for y, b in coef.iterrows():
        mask = p["form_year"] == y
        p.loc[mask, "Yhat"] = b["const"] + p.loc[mask, XCOLS].to_numpy(float) @ b[XCOLS].to_numpy(float)
    return p, coef


first_stage_coef = {}
for m in MEASURES:
    panels[m], first_stage_coef[m] = add_expected(panels[m])

fs_rows = {}
for m, c in first_stage_coef.items():
    r = {}
    for v in ["const"] + XCOLS:
        mu, t, _ = mean_t(c[v], NW_LAGS_ANNUAL)
        r[(v, "係数")] = mu
        r[(v, "t値")] = t
    r[("決定係数", "R2")] = c["R2"].mean()
    r[("決定係数", "adjR2")] = c["adjR2"].mean()
    r[("", "平均N")] = c["N"].mean()
    r[("", "年数")] = len(c)
    fs_rows[m] = r
first_stage_summary = pd.DataFrame(fs_rows).T
print("期待収益性の推定:年次回帰の係数の平均・t 値と決定係数")
display(first_stage_summary)

## 9. 銘柄が落ちる理由の検証

### 9-0. 検証の設定と段階の定義

毎年6月末の TOPIX 構成銘柄から出発し、分析と同じ順番の条件を **1つずつ積み重ねて** 残る銘柄数を数えます(ウォーターフォール)。

| 段階 | 条件(前の段階までの条件に加えて) | 対応する分析のコード |
|---|---|---|
| S0 | 6月末の TOPIX 構成銘柄(`get_topix_data` の有効期間) | `topix_tbl` |
| S1 | 証券コードに対応が付く(`get_nri_to_ticker_table`) | `attach_market` の `sec` |
| S2 | 業種コードがある(`get_industry`) | `attach_market` の `ind33` |
| S3 | 除外業種(電気・ガス・銀行・証券・保険・その他金融)でない | `EXCLUDE_IND33` |
| S4 | 6月末の時価総額がある(RISK_MODELS) | `ME_form` |
| S5 | その企業の本決算データ(12か月)が1件でもある | `acct` |
| S6 | その組成年に使える本決算がある(6月末までに公表、期末が15か月以内) | `form_year`・`stale_ok` |
| S7 | 自己資本 > 0 | `BE` |
| S8 | 決算期末の時価総額がある(B/M が計算できる) | `ME_fye`・`lnBM` |
| S9 | 収益性 Y が計算できる(前期の同じ会計基準の本決算がある、など) | 指標の値 |
| S10 | 収益性の変化 ΔY が計算できる(前々期まで必要) | `d指標` |
| S11 | 投資(資産・自己資本の成長率、t−2 → t−1)が計算できる | `inv` |
| S12 | 期待収益性 Yhat が計算できる(= 4-3 の「全体」) | `panels[指標]` で Y・Yhat がそろう銘柄 |

S7〜S11 は、その組成年に使える本決算(会計基準が複数ある場合はそのどれか)で条件を満たすものが1つでもあれば残ります(分析の `select_records` と同じ考え方)。

In [ ]:
OUTPUT_DIR = Path("outputs_universe_debug")
OUTPUT_DIR.mkdir(exist_ok=True)

DEBUG_MEASURES = MEASURES            # 検証する指標
SHOW_YEARS = [1995, 2000, 2003, 2010, 2015, 2020, 2024]   # 表に表示する組成年(データにない年は自動で除外)
EXAMPLE_YEAR = 2010                  # 落ちた銘柄の例を表示する組成年
N_EXAMPLES = 10                      # 理由ごとに表示する例の数

STEP_NAMES = {
    "S0": "S0 TOPIX構成銘柄",
    "S1": "S1 証券コードの対応あり",
    "S2": "S2 業種コードあり",
    "S3": "S3 除外業種でない",
    "S4": "S4 6月末の時価総額あり",
    "S5": "S5 本決算データが1件以上ある",
    "S6": "S6 その年に使える本決算あり",
    "S7": "S7 自己資本 > 0",
    "S8": "S8 決算期末の時価総額あり(B/M)",
    "S9": "S9 収益性 Y が計算できる",
    "S10": "S10 ΔY が計算できる",
    "S11": "S11 投資が計算できる",
    "S12": "S12 期待 Yhat が計算できる(最終)",
}

### 9-1. 6月末の TOPIX 構成銘柄と、市場データの段階(S0〜S4)

まず、取得した TOPIX 構成銘柄テーブルそのものの銘柄数を年ごとに確認します。
ここが実際の TOPIX の構成銘柄数より少なければ、テーブルの段階で銘柄が欠けていることになります。

In [ ]:
years = sorted(set(acct["form_year"]) | set(range(1987, int(acct["form_year"].max()) + 1)))
years = [y for y in years if y >= 1987]

base_rows = []
for y in years:
    F = formation_int(y)
    mem = topix_tbl.loc[(topix_tbl["FROM_DATE"] <= F) & (F <= topix_tbl["TO_DATE"]), "NRI_CODE"].unique()
    base_rows.append(pd.DataFrame({"NRI_CODE": mem, "form_year": y}))
base = pd.concat(base_rows, ignore_index=True)
base["F_int"] = base["form_year"].map(formation_int)
base = base.merge(asof_lookup(base, code_tbl, ["sec"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
base = base.merge(asof_lookup(base, ind_tbl, ["ind33"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
base["midx_form"] = base["form_year"] * 12 + FORMATION_MONTH - 1
base = base.merge(mcap.rename(columns={"midx": "midx_form", "me": "ME_form"}), on=["sec", "midx_form"], how="left")

base["S0"] = True
base["S1"] = base["S0"] & base["sec"].notna()
base["S2"] = base["S1"] & base["ind33"].notna()
base["S3"] = base["S2"] & ~base["ind33"].isin(EXCLUDE_IND33)
base["S4"] = base["S3"] & (base["ME_form"] > 0)

print("TOPIX 構成銘柄テーブルの、6月末時点の銘柄数(テーブルそのもの)")
display(base.groupby("form_year").size().to_frame("TOPIX構成銘柄数").T)

print("除外業種(S3)で落ちた銘柄の業種コード別の数(全年合計)")
display(base[base["S2"] & ~base["S3"]]["ind33"].value_counts().to_frame("銘柄年数").T)

### 9-2. 財務データの段階(S5〜S11)

その組成年に使える本決算(`form_year` が一致し、`stale_ok`)を候補として、条件を順に積み重ねます。

In [ ]:
cand = acct[acct["stale_ok"]].copy()
# 候補の決算に、組成日時点の証券コードで決算期末の時価総額を付ける
cand = cand.merge(base[["NRI_CODE", "form_year", "sec"]].drop_duplicates(), on=["NRI_CODE", "form_year"], how="inner")
cand = cand.merge(mcap.rename(columns={"midx": "fy_idx", "me": "ME_fye"}), on=["sec", "fy_idx"], how="left")
bm = cand["BE"] * ACCOUNTING_UNIT / cand["ME_fye"]
cand["bm_ok"] = bm > 0

firms_with_acct = set(acct["NRI_CODE"])
usable = set(zip(cand["NRI_CODE"], cand["form_year"]))
base["S5"] = base["S4"] & base["NRI_CODE"].isin(firms_with_acct)
base["S6"] = base["S5"] & pd.Series(list(zip(base["NRI_CODE"], base["form_year"]))).isin(usable).values


def any_by_firm_year(mask):
    """候補の決算のうち mask を満たすものが1つでもある (NRI_CODE, form_year) の集合"""
    return set(zip(cand.loc[mask, "NRI_CODE"], cand.loc[mask, "form_year"]))


key = pd.Series(list(zip(base["NRI_CODE"], base["form_year"])), index=base.index)
c7 = cand["BE"].notna()
c8 = c7 & cand["bm_ok"]
base["S7"] = base["S6"] & key.isin(any_by_firm_year(c7))
base["S8"] = base["S7"] & key.isin(any_by_firm_year(c8))

waterfalls = {}
for m in DEBUG_MEASURES:
    inv_col = "g_TA_lag" if SCALE[m] == "asset" else "g_BE_lag"
    c9 = c8 & cand[m].notna()
    c10 = c9 & cand["d" + m].notna()
    c11 = c10 & cand[inv_col].notna()
    b = base.copy()
    b["S9"] = b["S8"] & key.isin(any_by_firm_year(c9))
    b["S10"] = b["S9"] & key.isin(any_by_firm_year(c10))
    b["S11"] = b["S10"] & key.isin(any_by_firm_year(c11))
    final = panels[m].dropna(subset=["Y", "Yhat", "lnBM", "ME_form"])
    b["S12"] = b["S11"] & key.isin(set(zip(final["NRI_CODE"], final["form_year"])))
    waterfalls[m] = b

### 9-3. 段階ごとの銘柄数(ウォーターフォール)

各段階で **残っている銘柄数** と、**その段階で落ちた銘柄数** を年ごとに表示します。
最後の S12 は、`rmw18_factor_analysis.ipynb` の 4-3 の「全体」と一致するはずです。

In [ ]:
steps = list(STEP_NAMES)
show_years = [y for y in SHOW_YEARS if y in years]
wf_tables = {}
for m, b in waterfalls.items():
    remain = b.groupby("form_year")[steps].sum().T
    remain.index = [STEP_NAMES[s] for s in steps]
    dropped = (remain.shift(1) - remain).iloc[1:].astype(int)
    dropped.index = [f"落ちた:{i}" for i in dropped.index]
    wf_tables[m] = (remain, dropped)
    print(f"■ {m}:残っている銘柄数")
    display(remain[show_years])
    print(f"■ {m}:各段階で落ちた銘柄数")
    display(dropped[show_years])

# 4-3 との照合(最終段階 = panels の Y・Yhat がそろう銘柄)
check = pd.DataFrame({m: wf_tables[m][0].loc[STEP_NAMES["S12"]] for m in DEBUG_MEASURES})
check_panel = pd.DataFrame({m: panels[m].dropna(subset=["Y", "Yhat", "lnBM", "ME_form"]).groupby("form_year").size()
                            for m in DEBUG_MEASURES})
diff = (check.reindex(check_panel.index).fillna(0) - check_panel.fillna(0)).abs().to_numpy().sum()
print("照合:S12 と分析のパネル(4-3 の全体)の差の合計 =", diff, "(0 なら一致)")

### 9-4. 図:段階ごとに落ちた銘柄数の推移

In [ ]:
fig, axes = plt.subplots(len(DEBUG_MEASURES), 1, figsize=(14, 4.5 * len(DEBUG_MEASURES)), sharex=True)
axes = np.atleast_1d(axes)
for ax, m in zip(axes, DEBUG_MEASURES):
    remain, dropped = wf_tables[m]
    bottom = np.zeros(len(remain.columns))
    ax.bar(remain.columns, remain.loc[STEP_NAMES["S12"]], color="black", label="最終(S12)")
    bottom = remain.loc[STEP_NAMES["S12"]].to_numpy().astype(float)
    cmap = plt.get_cmap("tab20")
    for i, (name, row) in enumerate(dropped.iloc[::-1].iterrows()):
        ax.bar(remain.columns, row.to_numpy(), bottom=bottom, color=cmap(i), label=name.replace("落ちた:", ""))
        bottom = bottom + row.to_numpy()
    ax.plot(remain.columns, remain.loc[STEP_NAMES["S0"]], color="red", marker=".", label="TOPIX構成銘柄(S0)")
    ax.set_title(f"{m}:TOPIX 構成銘柄の内訳(黒 = 最終的に残った銘柄、色 = その段階で落ちた銘柄)")
    ax.set_ylabel("銘柄数")
axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "waterfall_by_year.png", dpi=120, bbox_inches="tight")
plt.show()

### 9-5. 「その年に使える本決算がない」(S6)の内訳

S6 で落ちた銘柄について、さらに理由を分けます。

| 内訳 | 意味 |
|---|---|
| 公表が6月末より後 | 組成日から15か月以内に期末を迎えた本決算はあるが、入手可能日(発表日・短信入手日)が6月末より後 |
| 期末が15か月より古い | その組成年に割り当てられた本決算はあるが、期末が古すぎる |
| 直近の本決算データなし | 組成日の前15か月に期末を迎えた本決算(12か月)がデータにない(変則決算・データ欠落など) |

入手可能日には「決算短信入手日」を使う場合があります。**古い年の入手日が、実際の公表日ではなくデータベースへの登録日になっている** と、
公表が遅れたように見えて銘柄が落ちます。下の表で、期末から入手可能日までの日数も確認します。

In [ ]:
b = waterfalls[DEBUG_MEASURES[0]]
s6_fail = b[b["S5"] & ~b["S6"]][["NRI_CODE", "form_year", "sec"]].copy()
s6_fail["F_midx"] = s6_fail["form_year"] * 12 + FORMATION_MONTH - 1
s6_fail["F_date"] = s6_fail["form_year"].map(formation_date)
rec = acct[["NRI_CODE", "fy_idx", "avail_date", "form_year"]].rename(columns={"form_year": "rec_form_year"})
mm = s6_fail.merge(rec, on="NRI_CODE", how="left")
mm["recent"] = (mm["fy_idx"] <= mm["F_midx"]) & (mm["F_midx"] - mm["fy_idx"] <= MAX_STALENESS_MONTHS)
mm["late"] = mm["recent"] & (mm["avail_date"] > mm["F_date"])
mm["assigned_stale"] = (mm["rec_form_year"] == mm["form_year"]) & ~mm["recent"]
agg = mm.groupby(["NRI_CODE", "form_year"]).agg(recent=("recent", "any"), late=("late", "any"),
                                                 stale=("assigned_stale", "any")).reset_index()
agg["内訳"] = np.select([agg["late"], agg["stale"]], ["公表が6月末より後", "期末が15か月より古い"], default="直近の本決算データなし")
if len(agg):
    s6_breakdown = agg.pivot_table(index="内訳", columns="form_year", values="NRI_CODE", aggfunc="count").fillna(0).astype(int)
else:
    s6_breakdown = pd.DataFrame()
print("S6 で落ちた銘柄の内訳")
display(s6_breakdown.reindex(columns=[y for y in show_years if y in s6_breakdown.columns]))

# 期末から入手可能日までの日数(決算期末の年ごと)
lag = acct.assign(lag_days=(acct["avail_date"] - acct["fdate"]).dt.days, fy_year=acct["fdate"].dt.year)
ann_ok = to_date(acct["ANNOUNCE"]) >= acct["fdate"]
rcv_ok = to_date(acct["RCVD"]) >= acct["fdate"]
lag["入手日の種類"] = np.select([ann_ok, rcv_ok], ["決算発表日", "短信入手日"], default="期末+2か月")
lag_summary = lag.groupby("fy_year").agg(本決算の件数=("lag_days", "size"), 日数の中央値=("lag_days", "median"),
                                         日数の90パーセンタイル=("lag_days", lambda x: x.quantile(0.9)),
                                         公表が期末から92日超の割合=("lag_days", lambda x: (x > 92).mean()))
src_share = lag.groupby("fy_year")["入手日の種類"].value_counts(normalize=True).unstack().fillna(0)
print("期末から入手可能日までの日数(決算期末の年ごと)と、入手日の種類の割合")
display(lag_summary.join(src_share).loc[lambda d: d.index >= 1985])

### 9-6. 「収益性 Y が計算できない」(S9)の内訳

S8 までは残ったが S9 で落ちた銘柄について、B/M まで計算できた決算(会計基準の優先順位で1つ)を見て、理由を分けます。

| 内訳 | 意味 |
|---|---|
| 前期の本決算なし(同じ会計基準) | 前期(期末 −12か月)に、同じ会計基準の本決算がない(上場直後、決算期変更、会計基準の変更など) |
| 分母が欠損・0以下 | 前期末の総資産または自己資本が欠損・0以下 |
| 分子の項目が欠損 | 営業利益・当期純利益などが欠損 |
| アクルーアルが計算できない | 売上債権・棚卸資産・買入債務が当期と前期の片方だけ欠損(CBOP・CP のみ) |

In [ ]:
s9_rows = []
for m in DEBUG_MEASURES:
    b = waterfalls[m]
    fail = b[b["S8"] & ~b["S9"]][["NRI_CODE", "form_year"]]
    c = cand[cand["BE"].notna() & cand["bm_ok"]].merge(fail, on=["NRI_CODE", "form_year"])
    c = c.sort_values(["NRI_CODE", "form_year", "fy_idx", "prio"], ascending=[True, True, False, True])
    c = c.drop_duplicates(["NRI_CODE", "form_year"])
    denom = c["TA_p"] if SCALE[m] == "asset" else c["BE_p"]
    num_col = {"CBOP": "OPINCM", "CP": "OPINCM", "ROE": "NETINCM", "ROA": "OPINCM"}[m]
    reason = np.select(
        [~c["prev_exists"], ~(denom > 0), c[num_col].isna(),
         c["accrual"].isna() if m in ("CBOP", "CP") else pd.Series(False, index=c.index)],
        ["前期の本決算なし(同じ会計基準)", "分母が欠損・0以下", "分子の項目が欠損", "アクルーアルが計算できない"],
        default="その他")
    s9_rows.append(pd.Series(reason).value_counts().rename(m))
s9_breakdown = pd.DataFrame(s9_rows).T.fillna(0).astype(int)
print("S9 で落ちた銘柄年数の内訳(全年合計)")
display(s9_breakdown)

### 9-7. 落ちた銘柄の例

`EXAMPLE_YEAR` の組成で落ちた銘柄を、段階ごとに `N_EXAMPLES` 件ずつ表示します。DB や開示資料で個別に確認するときに使ってください。

In [ ]:
m = DEBUG_MEASURES[0]
b = waterfalls[m]
b = b[b["form_year"] == EXAMPLE_YEAR]
ex_rows = []
for prev, cur in zip(steps[:-1], steps[1:]):
    d = b[b[prev] & ~b[cur]].head(N_EXAMPLES)
    for _, r in d.iterrows():
        ex_rows.append({"落ちた段階": STEP_NAMES[cur], "NRI_CODE": r["NRI_CODE"], "証券コード": r["sec"],
                        "業種コード": r["ind33"], "6月末時価総額(億円)": r["ME_form"] / 1e8})
examples = pd.DataFrame(ex_rows)
print(f"{EXAMPLE_YEAR} 年6月の組成で落ちた銘柄の例({m})")
display(examples)

### 9-8. 結果の保存

In [ ]:
for m, (remain, dropped) in wf_tables.items():
    pd.concat([remain, dropped]).to_csv(OUTPUT_DIR / f"waterfall_{m}.csv", encoding="utf-8-sig")
s6_breakdown.to_csv(OUTPUT_DIR / "s6_breakdown.csv", encoding="utf-8-sig")
lag_summary.join(src_share).to_csv(OUTPUT_DIR / "availability_lag.csv", encoding="utf-8-sig")
s9_breakdown.to_csv(OUTPUT_DIR / "s9_breakdown.csv", encoding="utf-8-sig")
examples.to_csv(OUTPUT_DIR / f"examples_{EXAMPLE_YEAR}.csv", encoding="utf-8-sig", index=False)
print(f"保存先:{OUTPUT_DIR.resolve()}")